# Agente de Preguntas y Respuestas con LangGraph y RAG

Notebook educativo en español para recorrer una ruta mínima de recuperación local (RAG) y luego extenderla con búsqueda web y una interfaz opcional de Gradio.


## Qué vas a aprender

- Instalar dependencias para Colab o Jupyter local.
- Cargar una base de conocimiento desde Wikipedia en español.
- Construir un grafo con `StateGraph`, `START` y `END`.
- Visualizar el flujo con Mermaid.
- Encender extensiones opcionales solo si las claves existen.


In [ ]:
%pip -q install langchain langgraph langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-groq chromadb tavily-python sentence-transformers gradio pandas python-dotenv ipywidgets


## 1) Claves API: seguro en Colab y en local

Este notebook no debe romperse si una clave no existe. Si una clave falta, se muestra un aviso claro y la ruta correspondiente queda en modo degradado.


In [ ]:
import os
from typing import TypedDict, List, Optional


def _load_colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None


def get_secret(name: str) -> Optional[str]:
    value = os.getenv(name)
    if value:
        return value
    value = _load_colab_secret(name)
    if value:
        os.environ[name] = value
    return value

def configure_optional_key(name: str) -> Optional[str]:
    value = get_secret(name)
    if value:
        print(f"{name}: configurada")
    else:
        print(f"AVISO: {name} no está disponible. Esa parte del notebook se ejecutará en modo degradado.")
    return value


os.environ.setdefault("USER_AGENT", "RAGAgentic/1.0")

groq_api_key = configure_optional_key("GROQ_API_KEY")
tavily_api_key = configure_optional_key("TAVILY_API_KEY")
langchain_api_key = configure_optional_key("LANGCHAIN_API_KEY")

if langchain_api_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ.setdefault("LANGCHAIN_PROJECT", "RAGAgentic - LangGraph educativo")
    os.environ.setdefault("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")
    print("LangSmith tracing habilitado.")
else:
    os.environ.pop("LANGCHAIN_TRACING_V2", None)
    print("LangSmith tracing desactivado.")


## 2) Base de conocimiento RAG

Usamos una sola fuente corta y estable para enseñar el flujo sin ruido: Wikipedia en español sobre LangChain.


In [ ]:
import pandas as pd
from IPython.display import display
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except Exception:
    from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

persist_directory = "./rag_db_colab_v2"
os.makedirs(persist_directory, exist_ok=True)
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = None
source_url = "https://es.wikipedia.org/wiki/LangChain"

try:
    if os.listdir(persist_directory):
        print(f"Cargando base vectorial existente desde '{persist_directory}'...")
        vector_store = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)
    else:
        raise FileNotFoundError
except Exception as exc:
    print(f"Creando base vectorial nueva en '{persist_directory}'...")
    try:
        loader = WebBaseLoader([source_url])
        docs = loader.load()
        print(f"Documentos cargados: {len(docs)}")
    except Exception as loader_exc:
        print(f"No se pudo leer Wikipedia; se usa un documento local de respaldo. Detalle: {loader_exc}")
        docs = [
            Document(
                page_content=(
                    "LangChain es un framework para construir aplicaciones con modelos de lenguaje. "
                    "Permite encadenar prompts, herramientas, memoria y recuperación de información. "
                    "RAG combina recuperación y generación para responder con contexto. "
                ),
                metadata={"source": "local-fallback", "title": "LangChain (respaldo local)", "language": "es"},
            )
        ]
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents(docs)
    print(f"Fragmentos generados: {len(splits)}")
    vector_store = Chroma.from_documents(documents=splits, embedding=embedding_function, persist_directory=persist_directory)

sample_docs = vector_store.similarity_search("¿Qué es LangChain?", k=3) if vector_store else []
if sample_docs:
    preview_df = pd.DataFrame([
        {
            "#": i + 1,
            "source": doc.metadata.get("source", ""),
            "title": doc.metadata.get("title", ""),
            "content": doc.page_content[:220].replace("\n", " ")
        }
        for i, doc in enumerate(sample_docs)
    ])
    display(preview_df)
else:
    print("No se pudo generar un preview de la base de conocimiento.")


## 3) Modelos y grader

El flujo usa Groq si la clave existe. Si no, el notebook sigue funcionando con respuestas y criterios heurísticos para no romper la enseñanza.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
from tavily import TavilyClient

GROQ_CHAT_MODEL = os.getenv("GROQ_CHAT_MODEL", "llama-3.1-70b-versatile")
GROQ_GRADER_MODEL = os.getenv("GROQ_GRADER_MODEL", "llama-3.1-8b-instant")

llm_groq = None
grader_llm = None
answer_prompt = ChatPromptTemplate.from_template(
    """Eres un profesor claro y breve.
Responde en español.

Pregunta: {question}

Contexto:
{context}

Si el contexto no alcanza, dilo con honestidad y sugiere el siguiente paso."""
)
retrieval_grader_prompt = ChatPromptTemplate.from_template(
    """Eres un evaluador experto.
Responde solo con JSON válido: {{"score": "yes" | "no"}}.

Pregunta:
{question}

Documento:
{document}"""
)

if groq_api_key:
    llm_groq = ChatGroq(model=GROQ_CHAT_MODEL, temperature=0.1)
    grader_llm = ChatGroq(model=GROQ_GRADER_MODEL, temperature=0)
    print(f"Groq listo con modelos: {GROQ_CHAT_MODEL} / {GROQ_GRADER_MODEL}")
else:
    print("Groq no está configurado. El notebook usará respuestas locales de respaldo.")

retrieval_grader = None
if grader_llm is not None:
    retrieval_grader = retrieval_grader_prompt | grader_llm | JsonOutputParser()

tavily_client = TavilyClient(api_key=tavily_api_key) if tavily_api_key else None
if tavily_client:
    print("Tavily listo.")
else:
    print("Tavily no está configurado. La búsqueda web quedará deshabilitada.")


## 4) Estado, nodos y lógica del grafo

Primero intentamos RAG local. Si falla o no alcanza, pasamos a web search solo cuando haga falta.


In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END


class GraphState(TypedDict):
    question: str
    documents: List[str]
    answer: str
    web_search_performed: bool


def format_docs(documents: List[str]) -> str:
    if not documents:
        return ""
    return "\n\n".join(f"[{i + 1}] {doc}" for i, doc in enumerate(documents))


def heuristic_relevance(question: str, document: str) -> str:
    q_words = {word for word in question.lower().split() if len(word) > 3}
    d_words = set(document.lower().split())
    return "yes" if len(q_words & d_words) >= 2 else "no"


def grade_document(question: str, document: str) -> str:
    if retrieval_grader is not None:
        try:
            result = retrieval_grader.invoke({"question": question, "document": document})
            score = str(result.get("score", "no")).lower()
            return "yes" if score == "yes" else "no"
        except Exception as exc:
            print(f"Aviso: el grader LLM falló y se usará una heurística local. Detalle: {exc}")
    return heuristic_relevance(question, document)


def retrieve_docs_node(state: GraphState) -> dict:
    print("--- Nodo: retrieve_docs_node ---")
    question = state["question"]
    if not vector_store:
        print("No hay vector_store disponible.")
        return {"documents": []}
    docs = vector_store.similarity_search(question, k=3)
    doc_contents = [doc.page_content for doc in docs]
    print(f"Recuperados {len(doc_contents)} fragmentos de RAG.")
    return {"documents": doc_contents}


def tavily_web_search_node(state: GraphState) -> dict:
    print("--- Nodo: tavily_web_search_node ---")
    question = state["question"]
    if tavily_client is None:
        print("Tavily no está configurado; no se realiza búsqueda web.")
        return {"documents": [], "web_search_performed": True}
    try:
        response = tavily_client.search(query=question, max_results=3)
        results = response.get("results", [])
        documents = []
        for item in results:
            title = item.get("title", "")
            content = item.get("content", "")
            url = item.get("url", "")
            documents.append(f"{title}\n{content}\nURL: {url}".strip())
        print(f"Se obtuvieron {len(documents)} resultados web.")
        return {"documents": documents, "web_search_performed": True}
    except Exception as exc:
        print(f"La búsqueda web falló: {exc}")
        return {"documents": [], "web_search_performed": True}


def generate_answer_node(state: GraphState) -> dict:
    print("--- Nodo: generate_answer_node ---")
    question = state["question"]
    context = format_docs(state.get("documents", [])) or "No hay contexto recuperado."

    if llm_groq is not None:
        try:
            chain = answer_prompt | llm_groq
            answer = chain.invoke({"question": question, "context": context}).content
            return {"answer": answer}
        except Exception as exc:
            print(f"Aviso: Groq falló; se usará respuesta local de respaldo. Detalle: {exc}")

    fallback = (
        "Modo local de respaldo.\n\n"
        f"Pregunta: {question}\n\n"
        f"Contexto:\n{context}\n\n"
        "Sugerencia: configura GROQ_API_KEY para activar la generación con LLM."
    )
    return {"answer": fallback}


def route_initial_question_logic(state: GraphState) -> str:
    question = state["question"]
    keywords_for_web = ["actualidad", "noticias", "última información", "reciente", "tiempo real", "hoy"]
    if any(keyword in question.lower() for keyword in keywords_for_web):
        print("Ruta inicial: web search.")
        return "web_search_needed"
    print("Ruta inicial: RAG local.")
    return "retrieve_first"


def grade_retrieved_documents_logic(state: GraphState) -> str:
    question = state["question"]
    documents = state.get("documents", [])
    if state.get("web_search_performed", False):
        print("Ya se realizó búsqueda web. Vamos a generar la respuesta.")
        return "generate_with_rag_docs"
    if not documents:
        print("No hay documentos RAG relevantes; pasamos a web search.")
        return "web_search_needed_after_rag"

    for index, document in enumerate(documents, start=1):
        score = grade_document(question, document)
        print(f"Documento {index}: {score}")
        if score == "yes":
            return "generate_with_rag_docs"

    print("Los documentos RAG no alcanzan; pasamos a web search.")
    return "web_search_needed_after_rag"


In [ ]:
workflow = StateGraph(GraphState)
workflow.add_node("retriever_node", retrieve_docs_node)
workflow.add_node("web_searcher_node", tavily_web_search_node)
workflow.add_node("answer_generator_node", generate_answer_node)

workflow.add_conditional_edges(
    START,
    route_initial_question_logic,
    {
        "retrieve_first": "retriever_node",
        "web_search_needed": "web_searcher_node",
    },
)

workflow.add_conditional_edges(
    "retriever_node",
    grade_retrieved_documents_logic,
    {
        "generate_with_rag_docs": "answer_generator_node",
        "web_search_needed_after_rag": "web_searcher_node",
    },
)

workflow.add_edge("web_searcher_node", "answer_generator_node")
workflow.add_edge("answer_generator_node", END)

app = workflow.compile()
print("Grafo compilado correctamente.")


## 5) Visualización del flujo

La representación primaria es Mermaid. La imagen PNG es opcional y nunca debe bloquear el notebook.


In [ ]:
from IPython.display import Markdown, Image, display

mermaid_text = app.get_graph().draw_mermaid()
display(Markdown("```mermaid\n" + mermaid_text + "\n```"))

try:
    png_data = app.get_graph().draw_mermaid_png()
    display(Image(png_data))
except Exception as exc:
    print(f"PNG opcional no disponible: {exc}")


## 6) Pruebas mínimas

Estas pruebas no requieren claves reales. Si Groq o Tavily no están configurados, verás la ruta de respaldo en lugar de un error.


In [ ]:
questions = [
    "¿Qué es LangChain según la base local?",
    "¿Cuál es la última información sobre Gemini 2.5 Flash?",
]

for i, question in enumerate(questions, start=1):
    print(f"\n--- Prueba {i} ---")
    result = app.invoke({"question": question, "documents": [], "answer": "", "web_search_performed": False})
    print(result.get("answer", "No se generó respuesta."))


## 7) Gradio opcional

La interfaz solo se lanza si `ENABLE_GRADIO=1`. Así evitamos que el notebook se quede abierto por accidente.


In [ ]:
try:
    import gradio as gr
except Exception:
    gr = None


def run_agent(question: str) -> str:
    if not question.strip():
        return "Escribe una pregunta primero."
    result = app.invoke({"question": question, "documents": [], "answer": "", "web_search_performed": False})
    return result.get("answer", "No se generó respuesta.")


if gr is None:
    print("Gradio no está instalado.")
elif app is None:
    print("El grafo no está disponible; no se lanza Gradio.")
elif os.getenv("ENABLE_GRADIO", "0") != "1":
    print("Gradio preparado, pero desactivado. Define ENABLE_GRADIO=1 para lanzarlo.")
else:
    iface = gr.Interface(
        fn=run_agent,
        inputs=gr.Textbox(lines=2, placeholder="Escribe tu pregunta aquí..."),
        outputs=gr.Textbox(),
        title="Agente RAG + LangGraph",
        description="Demo educativa con RAG local, búsqueda web opcional y respaldo local.",
    )
    iface.launch(share=True, debug=False)
